# Adversarial Paraphrasing on Colab

This notebook runs the simplified Colab-friendly path for this repository.

- Single `text-in / text-out` execution
- Runtime profile chosen from detected GPU memory
- Safe defaults for T4/L4/A100-class runtimes
- Fail-fast if the cloned repo does not contain the patched local entrypoints

If you want the exact Colab-friendly patches from this workspace, push this branch to your own fork first and update `REPO_URL` / `REPO_REF` in the next cell.

In [2]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/7Sageer/Adversarial-Paraphrasing.git"
REPO_REF = "main"
REPO_DIR = Path("/content/Adversarial-Paraphrasing")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if REPO_REF:
    subprocess.run(["git", "checkout", REPO_REF], check=True)

print("cwd:", Path.cwd())


cwd: /content/Adversarial-Paraphrasing


In [7]:
import subprocess
import sys
from pathlib import Path

requirements_path = Path("requirements-colab.txt")
if not requirements_path.exists():
    raise RuntimeError(
        "This notebook expects the patched Colab-friendly repo layout. "
        "Push this branch to your fork and set REPO_URL / REPO_REF in the previous cell."
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
    check=True,
)

# Install the missing 'cleantext' module
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "cleantext"],
    check=True,
)

print("Installed minimal Colab dependencies.")

Installed minimal Colab dependencies.


In [3]:
import json
import subprocess
import sys
from pathlib import Path


def detect_gpu():
    nvidia_smi = Path("/usr/bin/nvidia-smi")
    if not nvidia_smi.exists():
        return {
            "name": "CPU",
            "memory_gb": 0.0,
            "device": "cpu",
            "precision": "float32",
        }

    raw = subprocess.check_output(
        [
            str(nvidia_smi),
            "--query-gpu=name,memory.total",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    ).strip().splitlines()[0]
    name, memory_mb = [part.strip() for part in raw.split(",", 1)]
    return {
        "name": name,
        "memory_gb": round(int(memory_mb) / 1024, 1),
        "device": "cuda",
        "precision": "float16",
    }


def choose_profile(gpu):
    memory_gb = gpu["memory_gb"]

    if memory_gb == 0:
        return {
            "label": "cpu-safe",
            "model": "Qwen/Qwen2.5-0.5B-Instruct",
            "guidance_classifier": "openai_roberta_base",
            "deploy_classifier": "openai_roberta_base",
            "adversarial": 0.0,
            "batch_size": 1,
            "max_new_tokens": 64,
            "note": "CPU only. Use non-adversarial smoke tests first.",
        }

    if memory_gb <= 16.5:
        return {
            "label": "t4-safe",
            "model": "Qwen/Qwen2.5-0.5B-Instruct",
            "guidance_classifier": "openai_roberta_base",
            "deploy_classifier": "openai_roberta_base",
            "adversarial": 0.0,
            "batch_size": 1,
            "max_new_tokens": 96,
            "note": "T4/P100-class profile. Start without adversarial search, then enable it if runtime is stable.",
        }

    if memory_gb <= 25:
        return {
            "label": "l4-balanced",
            "model": "Qwen/Qwen2.5-1.5B-Instruct",
            "guidance_classifier": "openai_roberta_base",
            "deploy_classifier": "openai_roberta_base",
            "adversarial": 1.0,
            "batch_size": 1,
            "max_new_tokens": 160,
            "note": "L4/A10-class profile. Detector-guided mode is reasonable here.",
        }

    return {
        "label": "a100-roomy",
        "model": "Qwen/Qwen2.5-3B-Instruct",
        "guidance_classifier": "openai_roberta_base",
        "deploy_classifier": "openai_roberta_base",
        "adversarial": 1.0,
        "batch_size": 1,
        "max_new_tokens": 192,
        "note": "40GB+ GPU profile. Still uses a conservative model size for notebook reliability.",
    }


gpu = detect_gpu()
profile = choose_profile(gpu)
print(json.dumps({"gpu": gpu, "profile": profile}, indent=2))


{
  "gpu": {
    "name": "Tesla T4",
    "memory_gb": 15.0,
    "device": "cuda",
    "precision": "float16"
  },
  "profile": {
    "label": "t4-safe",
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "guidance_classifier": "openai_roberta_base",
    "deploy_classifier": "openai_roberta_base",
    "adversarial": 0.0,
    "batch_size": 1,
    "max_new_tokens": 96,
    "note": "T4/P100-class profile. Start without adversarial search, then enable it if runtime is stable."
  }
}


## Profile Notes

- `T4 / P100 / 16GB-ish`: keep `Qwen2.5-0.5B`, `batch_size=1`, and start with `adversarial=0`
- `L4 / A10 / 24GB-ish`: `Qwen2.5-1.5B` with `openai_roberta_base` is the default balanced path
- `A100 / 40GB+`: the notebook still defaults to `3B` for reliability; the original paper-scale setup is still a separate problem
- If you see OOMs, reduce `max_new_tokens` first, then fall back to a smaller model


In [4]:
INPUT_TEXT = """Large language models can generate fluent text quickly, but many detectors still rely on subtle statistical regularities."""

CONFIG = {
    "model": profile["model"],
    "guidance_classifier": profile["guidance_classifier"],
    "deploy_classifier": profile["deploy_classifier"],
    "device": gpu["device"],
    "precision": gpu["precision"],
    "batch_size": profile["batch_size"],
    "max_new_tokens": profile["max_new_tokens"],
    "top_p": 0.9,
    "top_k": 40,
    "adversarial": profile["adversarial"],
}
CONFIG


{'model': 'Qwen/Qwen2.5-0.5B-Instruct',
 'guidance_classifier': 'openai_roberta_base',
 'deploy_classifier': 'openai_roberta_base',
 'device': 'cuda',
 'precision': 'float16',
 'batch_size': 1,
 'max_new_tokens': 96,
 'top_p': 0.9,
 'top_k': 40,
 'adversarial': 0.0}

In [8]:
import subprocess
import sys

# Try to get help text and check for errors
help_cmd = [sys.executable, "paraphrase_and_detect.py", "--help"]
help_result = subprocess.run(help_cmd, text=True, capture_output=True)

if help_result.returncode != 0:
    print("Error getting help text from paraphrase_and_detect.py:")
    if help_result.stdout:
        print("STDOUT:", help_result.stdout)
    if help_result.stderr:
        print("STDERR:", help_result.stderr)
    raise RuntimeError(f"Failed to execute '{' '.join(help_cmd)}'. Check the output above for details.")

help_text = help_result.stdout

if "--input_text" not in help_text:
    raise RuntimeError(
        "The cloned repo does not contain the patched Colab-friendly entrypoint. "
        "Push this branch to your fork and update REPO_URL / REPO_REF."
    )

cmd = [
    sys.executable,
    "paraphrase_and_detect.py",
    "--input_text",
    INPUT_TEXT,
    "--model",
    CONFIG["model"],
    "--guidance_classifier",
    CONFIG["guidance_classifier"],
    "--deploy_classifier",
    CONFIG["deploy_classifier"],
    "--device",
    CONFIG["device"],
    "--precision",
    CONFIG["precision"],
    "--batch_size",
    str(CONFIG["batch_size"]),
    "--max_new_tokens",
    str(CONFIG["max_new_tokens"]),
    "--top_p",
    str(CONFIG["top_p"]),
    "--top_k",
    str(CONFIG["top_k"]),
    "--adversarial",
    str(CONFIG["adversarial"]),
]

print("Running:")
print(" ".join(cmd[:2]), "...")
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    raise RuntimeError(f"Notebook command failed with exit code {result.returncode}")

Running:
/usr/bin/python3 paraphrase_and_detect.py ...
******************** 
 Namespace(dataset='mage', input_text='Large language models can generate fluent text quickly, but many detectors still rely on subtle statistical regularities.', model='Qwen/Qwen2.5-0.5B-Instruct', guidance_classifier='openai_roberta_base', deploy_classifier='openai_roberta_base', device='cuda', precision='float16', watermark_tokenizer='', batch_size=1, num_samples=10, top_p=0.9, top_k=40, adversarial=0.0, option=None, deterministic=1, n_words_sample=100, max_new_tokens=96, debug=0, hf_cache_dir='') 
 ******************** 

Loading deploy classifier...
Loaded 1 texts.



********************
<input>Large language models can generate fluent text quickly, but many detectors still rely on subtle statistical regularities.</input>
<inp_score>0.1343160718679428</inp_score>

<output>Large language models can generate fluent text quickly, but many detectors still rely on subtle statistical regularities.</output>
<out